In [1]:
print("c")

c


In [14]:
import os
from dotenv import load_dotenv

import pandas as pd
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

True

In [15]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [17]:
import os
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:llama-3.3-70b-versatile")
response=model.invoke("Why am I learning at DoT")
response

AIMessage(content='I\'m not sure about your specific situation, but I can provide some general insights about learning at the "Department of Transportation" (DoT) or similar organizations.\n\nPeople might learn at the DoT for various reasons, such as:\n\n1. **Professional development**: Employees of the DoT might engage in training and learning to improve their skills, stay updated with the latest regulations, and advance in their careers.\n2. **Regulatory compliance**: The DoT has various rules and regulations to ensure public safety, and learning about these regulations is essential for professionals working in the transportation industry.\n3. **Industry-specific knowledge**: The DoT provides a wealth of information on transportation-related topics, such as infrastructure development, traffic management, and vehicle safety standards.\n4. **Certification and licensure**: Some professionals might learn at the DoT to obtain certifications or licenses required for their jobs, such as com

In [18]:
class AnimeDataLoader:
    def __init__(self, original_csv: str, processed_csv: str):
        self.original_csv = original_csv
        self.processed_csv = processed_csv

    def load_and_process(self):
        df = pd.read_csv(
            self.original_csv,
            encoding="utf-8",
            on_bad_lines="skip"
        ).dropna()

        required_cols = {"Name", "Genres", "sypnopsis"}
        if not required_cols.issubset(df.columns):
            raise ValueError("Missing required columns in CSV")

        df["combined_info"] = (
            "Title: " + df["Name"]
            + " Overview: " + df["sypnopsis"]
            + " Genres: " + df["Genres"]
        )

        df[["combined_info"]].to_csv(
            self.processed_csv,
            index=False,
            encoding="utf-8"
        )

        return self.processed_csv


In [20]:
og_csv="data/anime_with_synopsis.csv"
procssed_csv="data/anime_updated.csv"
main_class_call_karrahi_hu=AnimeDataLoader(og_csv,procssed_csv)
#calling the method
main_class_call_karrahi_hu.load_and_process()

'data/anime_updated.csv'

DATA SPLITTING

In [21]:
class VectorStoreBuilder: # its a class
    def __init__(self, csv_path:str, persist_dir:str="ChromaDB"): #initialising a class, #persist dir tells the user tell me ki kahan store karna hai
        self.csv_path=csv_path 
        self.persist_dir=persist_dir
        #3 types of memory: session memory aur persistent storage (stored in a folder), cloud directory
        self.embedding=HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2" #embedding numerical values me convert karta hai
        )                                                    # default name of the folder is "chroma db"
        pass
    def build_and_save_vectorstore(self): #vectore build aur save karega
        loader = CSVLoader(
            file_path=self.csv_path,
            encoding="utf-8",
            metadata_columns=[]
        )
        documents=loader.load()
        splitter=RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=20
        ) #this is a class
        chunks=splitter.split_documents(documents)

        #database banayenge abhi, chroma naam ka class hai isme method hai from documents
        db=Chroma.from_documents(
            chunks,
            self.embedding,
            persist_directory=self.persist_dir
        )
        db.persist()
    def load_vector_store(self):#vectorstore load bhi karta hai
        return Chroma(
            persist_directory=self.persist_dir,
            embedding_function=self.embedding
        )



In [23]:
#calling upar ke methods
print("BUILDING A VECTOR STORE")
og_csv="data/anime_with_synopsis.csv"
procssed_csv="data/anime_updated.csv"
persist_dir="ChromaDB"
data_loader=AnimeDataLoader(og_csv,procssed_csv)
processed_csv_path = data_loader.load_and_process()

BUILDING A VECTOR STORE


In [24]:
print("STILL BUILDING A VECTOR STORE")
vector_builder = VectorStoreBuilder(
    processed_csv_path,
    persist_dir=persist_dir
)
vector_builder.build_and_save_vectorstore()

print("Vector database ready.")

STILL BUILDING A VECTOR STORE
Vector database ready.


/var/folders/75/g_sfzqxn0gn9p6625cdcl4dm0000gn/T/ipykernel_37138/3803467310.py:29: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


retriever tool

In [25]:
vector_builder = VectorStoreBuilder(
    csv_path="",
    persist_dir="ChromaDB"
)
retriever=vector_builder.load_vector_store().as_retriever()


/var/folders/75/g_sfzqxn0gn9p6625cdcl4dm0000gn/T/ipykernel_37138/3803467310.py:31: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


In [26]:
query="tell me about naruto"
retriever.invoke(query)

[Document(metadata={'row': 10, 'source': 'data/anime_updated.csv'}, page_content="combined_info: Title: Naruto Overview: oments prior to Naruto Uzumaki's birth, a huge demon known as the Kyuubi, the Nine-Tailed Fox, attacked Konohagakure, the Hidden Leaf Village, and wreaked havoc. In order to put an end to the Kyuubi's rampage, the leader of the village, the Fourth Hokage, sacrificed his life and sealed the monstrous beast inside the newborn Naruto. Now, Naruto is a hyperactive and knuckle-headed ninja still living in Konohagakure. Shunned because of the Kyuubi inside him, Naruto struggles to find his place in the village, while his burning desire to become the Hokage of Konohagakure leads him not only to some great new friends, but also some deadly foes. Genres: Action, Adventure, Comedy, Super Power, Martial Arts, Shounen"),
 Document(metadata={'source': 'data/anime_updated.csv', 'row': 43}, page_content='Kotori, the two sisters Mizukoshi Moe and Mako who prefer eating nabe on the s

In [27]:
@tool #decorator class
def anime_retriever_tool(query:str)->str: #write a docstring for any tool or function
    """
    Use this tool to search the anime knowledge base.

    Always call this tool for anime-related questions such as:
    recommendations, similarity search, genres, or plot summaries.

    Input:
    - query: User's anime preference or question.

    Output:
    - Relevant anime information retrieved from the vector database.
    """
    docs=retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

In [28]:
model_with_tools=model.bind_tools([anime_retriever_tool])

In [29]:
messages=[
    {
        "role":"user",
        "content":"can you suggest something similar to naruto and also tell me a joke"
    }
]

In [32]:
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)
messages


[{'role': 'user',
  'content': 'can you suggest something similar to naruto and also tell me a joke'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6hd7kf4am', 'function': {'arguments': '{"query":"similar to Naruto"}', 'name': 'anime_retriever_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 341, 'total_tokens': 382, 'completion_time': 0.099552029, 'completion_tokens_details': None, 'prompt_time': 0.032089346, 'prompt_tokens_details': None, 'queue_time': 0.053356373, 'total_time': 0.131641375}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c1354-4afb-7f30-b07e-9ebf9c66df48-0', tool_calls=[{'name': 'anime_retriever_tool', 'args': {'query': 'similar to Naruto'}, 'id': '6hd7kf4am', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3

In [33]:
ai_msg.tool_calls

[]

In [34]:
messages[0]

{'role': 'user',
 'content': 'can you suggest something similar to naruto and also tell me a joke'}

In [35]:
for tool_call in ai_msg.tool_calls:
    tool_result=anime_retriever_tool.invoke(tool_call)
    messages.append(tool_result)
messages

[{'role': 'user',
  'content': 'can you suggest something similar to naruto and also tell me a joke'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6hd7kf4am', 'function': {'arguments': '{"query":"similar to Naruto"}', 'name': 'anime_retriever_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 341, 'total_tokens': 382, 'completion_time': 0.099552029, 'completion_tokens_details': None, 'prompt_time': 0.032089346, 'prompt_tokens_details': None, 'queue_time': 0.053356373, 'total_time': 0.131641375}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c1354-4afb-7f30-b07e-9ebf9c66df48-0', tool_calls=[{'name': 'anime_retriever_tool', 'args': {'query': 'similar to Naruto'}, 'id': '6hd7kf4am', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3

In [37]:
final_response = model_with_tools.invoke(messages)
print(final_response.text)

In [38]:
# Step 1: Model decides tool usage
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools
for tool_call in ai_msg.tool_calls:
    tool_result = anime_retriever_tool.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


Here are some anime suggestions similar to Naruto: 

1. Bleach - The story follows Ichigo Kurosaki, a high school student who becomes a Soul Reaper, tasked with protecting humans from evil spirits.
2. One Piece - The series follows Monkey D. Luffy and his crew, the Straw Hat Pirates, as they search for the ultimate treasure known as "One Piece" and face various enemies and challenges along the way.
3. Fairy Tail - The story takes place in a world where magic is a part of everyday life, and follows Lucy, a young wizard, and her friends in the magical guild Fairy Tail as they go on adventures and battle against evil forces.

I hope you find something you like! 

As for the joke, I hope you found it amusing!
